# Question 2 - Logistic Regression (Homework 1)

**Why this cell exists:** It states scope and maps this notebook to the HW document.

This notebook is the solution for **Question 2 (Logistic Regression)** from `Homework1.pdf`.
It is organized for presentation, with each code block preceded by markdown that explains:
- what the block does,
- why it exists in the HW flow,
- and the formulas used in that block.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Config
# -----------------------------
DATA_PATH = "bank-full.csv"
RANDOM_STATE = 42
TEST_SIZE = 0.2

# As required in HW 2.3 (multiple learning rates and epochs)
LEARNING_RATES = [0.1, 0.05, 0.01]
EPOCHS_LIST = [300, 600]

# Numerical stability for logs in cross-entropy
EPS = 1e-12

## Core Math Utilities

Key formulas used here:

1. Logistic model (binary classification):
$$
z = b + \mathbf{w}^T\mathbf{x}, \quad p(y=1\mid x)=\sigma(z)=\frac{1}{1+e^{-z}}
$$

2. Binary cross-entropy loss:
$$
J(\theta) = -\frac{1}{m}\sum_{i=1}^{m}\left[y^{(i)}\log p^{(i)} + (1-y^{(i)})\log(1-p^{(i)})\right]
$$

3. Confusion matrix structure:
$$
\begin{bmatrix}
TN & FP \\
FN & TP
\end{bmatrix}
$$

4. Metrics:
$$
\text{Accuracy}=\frac{TP+TN}{TP+TN+FP+FN},\quad
\text{Precision}=\frac{TP}{TP+FP},\quad
\text{Recall}=\frac{TP}{TP+FN},\quad
F1=\frac{2PR}{P+R}
$$


In [24]:
def set_seed(seed: int):
    np.random.seed(seed)


def sigmoid(z: np.ndarray) -> np.ndarray:
    # Numerically stable sigmoid
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))


def binary_cross_entropy(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    y_prob = np.clip(y_prob, EPS, 1 - EPS)
    m = y_true.shape[0]
    return float(-(1.0 / m) * np.sum(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob)))


def confusion_matrix_binary(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    return np.array([[tn, fp], [fn, tp]], dtype=int)


def classification_metrics(y_true: np.ndarray, y_pred: np.ndarray):
    cm = confusion_matrix_binary(y_true, y_pred)
    tn, fp, fn, tp = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]
    n = tn + fp + fn + tp

    accuracy = (tp + tn) / n if n else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0
    return accuracy, precision, recall, f1, cm


def explain_metric_definitions():
    print("Metric definitions and interpretation:")
    print("  Accuracy  = (TP + TN) / (TP + TN + FP + FN)")
    print("    -> Overall fraction of correct predictions.")
    print("  Precision = TP / (TP + FP)")
    print("    -> Of predicted positives, how many are truly positive (controls false alarms).")
    print("  Recall    = TP / (TP + FN)")
    print("    -> Of actual positives, how many are detected (controls misses).")
    print("  F1-score  = 2 * (Precision * Recall) / (Precision + Recall)")
    print("    -> Harmonic balance of precision and recall.")


## Split And Feature Scaling

**Why this cell exists:** The HW requires train/test evaluation and proper preprocessing.

1. **Stratified split** preserves class proportions in train/test.
2. **Standardization** is applied using train statistics only:
$$
x' = \frac{x-\mu_{train}}{\sigma_{train}}
$$
This prevents leakage from test data into training.


In [25]:
def stratified_train_test_split(X: np.ndarray, y: np.ndarray, test_size: float, seed: int):
    rng = np.random.default_rng(seed)
    idx0 = np.where(y == 0)[0]
    idx1 = np.where(y == 1)[0]

    rng.shuffle(idx0)
    rng.shuffle(idx1)

    n0_test = int(round(len(idx0) * test_size))
    n1_test = int(round(len(idx1) * test_size))

    test_idx = np.concatenate([idx0[:n0_test], idx1[:n1_test]])
    train_idx = np.concatenate([idx0[n0_test:], idx1[n1_test:]])

    rng.shuffle(train_idx)
    rng.shuffle(test_idx)

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


def standardize_train_test(X_train: np.ndarray, X_test: np.ndarray, continuous_mask: np.ndarray):
    X_train_std = X_train.copy()
    X_test_std = X_test.copy()

    means = np.zeros(X_train.shape[1], dtype=float)
    stds = np.ones(X_train.shape[1], dtype=float)

    cols = np.where(continuous_mask)[0]
    for j in cols:
        mu = float(np.mean(X_train[:, j]))
        sigma = float(np.std(X_train[:, j], ddof=0))
        if sigma < 1e-12:
            sigma = 1.0
        means[j] = mu
        stds[j] = sigma
        X_train_std[:, j] = (X_train[:, j] - mu) / sigma
        X_test_std[:, j] = (X_test[:, j] - mu) / sigma

    return X_train_std, X_test_std, means, stds


## Logistic Regression From Scratch

**Why this cell exists:** This is the core model required by HW 2.3 (no ML model library).

Model and gradient used in batch gradient descent:

$$
\mathbf{X}_b=[\mathbf{1},\mathbf{X}],\quad \mathbf{p}=\sigma(\mathbf{X}_b\mathbf{w})
$$

$$
\nabla_{\mathbf{w}}J = \frac{1}{m}\mathbf{X}_b^T(\mathbf{p}-\mathbf{y}),\quad
\mathbf{w} \leftarrow \mathbf{w} - \alpha\nabla_{\mathbf{w}}J
$$


In [26]:
class LogisticRegressionScratch:
    def __init__(self, lr=0.01, epochs=500):
        self.lr = float(lr)
        self.epochs = int(epochs)
        self.w = None  # includes bias as w[0]

    def fit(self, X: np.ndarray, y: np.ndarray):
        m, d = X.shape
        Xb = np.hstack([np.ones((m, 1)), X])
        self.w = np.zeros(d + 1, dtype=float)

        losses = []
        for _ in range(self.epochs):
            z = Xb @ self.w
            p = sigmoid(z)
            loss = binary_cross_entropy(y, p)
            losses.append(loss)

            grad = (1.0 / m) * (Xb.T @ (p - y))
            self.w -= self.lr * grad

        return np.array(losses, dtype=float)

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        m = X.shape[0]
        Xb = np.hstack([np.ones((m, 1)), X])
        return sigmoid(Xb @ self.w)

    def predict(self, X: np.ndarray, threshold: float = 0.5) -> np.ndarray:
        return (self.predict_proba(X) >= threshold).astype(int)


## Data Loading, Encoding, And Feature Selection

**Why this cell exists:** Logistic regression needs numeric inputs, and HW 2.2 asks for exploratory preparation.

- Binary categorical variables are mapped to 0/1.
- Nominal categorical variables are one-hot encoded.
- `education` is ordinal-encoded when meaningful.
- Top-7 features are selected by absolute Pearson correlation with target:
$$
|\text{corr}(x_j, y)|
$$


In [27]:
def load_bank_data(path: str) -> pd.DataFrame:
    # bank-full is typically ';' separated
    try:
        df = pd.read_csv(path, sep=";")
    except Exception:
        df = pd.read_csv(path)
    return df


def encode_features(df: pd.DataFrame, target_col: str = "y"):
    df = df.copy()

    # target
    if df[target_col].dtype == object:
        df[target_col] = df[target_col].map({"no": 0, "yes": 1})
    y = df[target_col].to_numpy(dtype=int)

    Xdf = df.drop(columns=[target_col])

    # detect object columns
    obj_cols = [c for c in Xdf.columns if Xdf[c].dtype == object]

    # map yes/no binary categorical columns
    for c in obj_cols:
        vals = set(Xdf[c].dropna().unique().tolist())
        if vals.issubset({"yes", "no"}):
            Xdf[c] = Xdf[c].map({"no": 0, "yes": 1})

    # ordinal encode education where meaningful
    if "education" in Xdf.columns and Xdf["education"].dtype == object:
        order = {
            "primary": 0,
            "secondary": 1,
            "tertiary": 2,
            "unknown": -1,
        }
        Xdf["education"] = Xdf["education"].map(lambda v: order.get(str(v).strip().lower(), -1))

    # refresh object columns
    obj_cols = [c for c in Xdf.columns if Xdf[c].dtype == object]

    # one-hot encode nominal categorical columns
    if len(obj_cols) > 0:
        Xdf = pd.get_dummies(Xdf, columns=obj_cols, drop_first=False)

    X = Xdf.to_numpy(dtype=float)
    feature_names = Xdf.columns.tolist()
    return X, y, feature_names, Xdf


def top_k_correlated_features(Xdf: pd.DataFrame, y: np.ndarray, k: int = 7):
    y_series = pd.Series(y, name="target")
    corrs = {}
    for col in Xdf.columns:
        c = pd.Series(Xdf[col]).corr(y_series)
        if pd.isna(c):
            c = 0.0
        corrs[col] = float(c)

    sorted_cols = sorted(corrs.keys(), key=lambda c: abs(corrs[c]), reverse=True)
    selected = sorted_cols[:k]
    return selected, corrs


## Plotting Helpers

**Why this cell exists:** HW 2.3 explicitly asks for loss curves and confusion matrix plotting.

This cell provides reusable plotting functions for:
- standardized feature histograms,
- loss vs epoch,
- confusion matrix heatmap.


In [28]:
def plot_histograms(X_train: np.ndarray, feature_names: list, continuous_mask: np.ndarray, out_prefix="hist"):
    idxs = np.where(continuous_mask)[0]
    if len(idxs) == 0:
        print("No continuous numerical features detected for histogram plotting.")
        return

    for j in idxs:
        plt.figure()
        plt.hist(X_train[:, j], bins=30)
        plt.title(f"Histogram (standardized): {feature_names[j]}")
        plt.xlabel("Value")
        plt.ylabel("Count")
        plt.tight_layout()
        plt.savefig(f"{out_prefix}_{feature_names[j].replace('/', '_')}.png", dpi=150)
        plt.close()


def plot_confusion_matrix(cm: np.ndarray, out_path="confusion_matrix.png"):
    plt.figure()
    plt.imshow(cm, interpolation="nearest")
    plt.title("Confusion Matrix (Test Set)")
    plt.colorbar()
    tick_marks = np.arange(2)
    plt.xticks(tick_marks, ["Pred 0", "Pred 1"])
    plt.yticks(tick_marks, ["True 0", "True 1"])

    for i in range(2):
        for j in range(2):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center")

    plt.tight_layout()
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_loss_curves(results, out_path="loss_curves.png"):
    plt.figure()
    for label, losses in results:
        plt.plot(np.arange(len(losses)), losses, label=label)
    plt.title("Training Loss (Cross-Entropy) vs Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


- **Convergence speed metric**: first epoch where 90% of total loss drop is reached.
- **Parameter role explanation** uses odds ratio for each coefficient:
$$
\text{odds multiplier} = e^{w_i}
$$
- **Confusion matrix importance** reports error-type rates:
$$
\text{FPR}=\frac{FP}{FP+TN},\quad \text{FNR}=\frac{FN}{FN+TP}
$$


In [35]:
def convergence_epoch_to_90pct(losses: np.ndarray) -> int:
    if len(losses) == 0:
        return 0
    start = float(losses[0])
    end = float(losses[-1])
    target = start - 0.9 * (start - end)
    idx = np.where(losses <= target)[0]
    return int(idx[0] + 1) if len(idx) else int(len(losses))


def explain_parameter_roles(feature_names: list, weights: np.ndarray, continuous_mask: np.ndarray):
    bias = float(weights[0])
    print("Parameter-role interpretation (logistic model):")
    print("  logit(p) = bias + sum(w_i * x_i)")
    print(f"  bias = {bias:+.6f}: baseline log-odds when all input features are 0.")

    for j, (name, w) in enumerate(zip(feature_names, weights[1:])):
        w = float(w)
        direction = "increases" if w > 0 else ("decreases" if w < 0 else "does not change")
        odds_mult = float(np.exp(w))
        if continuous_mask[j]:
            unit_note = "+1 standardized unit (= +1 std in the original feature)"
        else:
            unit_note = "+1 encoded unit"
        print(
            f"  - {name}: w={w:+.6f} -> {unit_note} {direction} log-odds; "
            f"odds are multiplied by {odds_mult:.3f}."
        )


def explain_confusion_matrix_importance(cm: np.ndarray):
    tn, fp, fn, tp = int(cm[0, 0]), int(cm[0, 1]), int(cm[1, 0]), int(cm[1, 1])
    total = tn + fp + fn + tp
    fpr = (fp / (fp + tn)) if (fp + tn) else 0.0
    fnr = (fn / (fn + tp)) if (fn + tp) else 0.0

    print("Confusion matrix interpretation and importance:")
    print(f"  TN={tn}, FP={fp}, FN={fn}, TP={tp}, Total={total}")
    print(f"  False Positive Rate (FP/(FP+TN)) = {fpr:.4f}")
    print(f"  False Negative Rate (FN/(FN+TP)) = {fnr:.4f}")
    print(
        "  Importance: unlike a single score, the confusion matrix shows error types "
        "(false alarms vs missed positives), which is critical for threshold tuning "
        "and application-specific cost trade-offs."
    )


## Step A - Dataset Understanding (HW 2.2)

**Why this cell exists:** HW 2.2 asks to inspect the dataset and class distribution before modeling.

This cell prints:
- total samples/features,
- data types (`info()`),
- class counts and proportions,
- a simple class-distribution plot.


In [30]:
set_seed(RANDOM_STATE)

df = load_bank_data(DATA_PATH)
print("Loaded dataset:", df.shape)
print("Columns:", list(df.columns))

print("Total samples:", df.shape[0])
print("Total features (incl. target):", df.shape[1])

print("Data info:")
df.info()

TARGET_COL = "y"
counts = df[TARGET_COL].value_counts(dropna=False)
proportions = df[TARGET_COL].value_counts(normalize=True, dropna=False)

print("Class counts:")
print(counts)
print("Class proportions:")
print((proportions * 100).round(2).astype(str) + "%")

if min(proportions) / max(proportions) < 0.5:
    print("-> This looks IMBALANCED (heuristic: minority < 50% of majority).")
else:
    print("-> This looks relatively BALANCED (heuristic).")

plt.figure()
counts.plot(kind="bar")
plt.title("Class Distribution of Target (y)")
plt.xlabel("Class")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("class_distribution_y.png", dpi=150)
plt.close()
print("Saved class distribution plot: class_distribution_y.png")


Loaded dataset: (45211, 17)
Columns: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']
Total samples: 45211
Total features (incl. target): 17
Data info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  in

## Step B - Encoding, Feature Selection, Split, Scaling

**Why this cell exists:** It prepares the exact train/test data used by the logistic model.

This cell does:
1. categorical encoding,
2. top-7 feature selection by absolute correlation,
3. stratified 80/20 split,
4. duplicate removal in train (if any),
5. standardization for continuous selected features,
6. histogram plotting for standardized continuous features.


In [31]:
# Encode categorical features
X, y, feature_names, Xdf_encoded = encode_features(df, target_col="y")
print("After encoding:")
print("X shape:", X.shape, "y shape:", y.shape)
print("Num features after encoding:", X.shape[1])

# Select top-7 correlated features
selected_cols, corrs = top_k_correlated_features(Xdf_encoded, y, k=7)
print("Top-7 features by |correlation| with target:")
for c in selected_cols:
    print(f"  {c:30s} corr={corrs[c]:+.4f}")

Xdf_sel = Xdf_encoded[selected_cols].copy()
X_all = Xdf_sel.to_numpy(dtype=float)

# Stratified split
X_train, X_test, y_train, y_test = stratified_train_test_split(
    X_all, y, test_size=TEST_SIZE, seed=RANDOM_STATE
)
print("Stratified split shapes:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

# Remove duplicates in train (based on selected features + label)
train_df_tmp = pd.DataFrame(X_train, columns=selected_cols)
train_df_tmp["y"] = y_train
dup_mask = train_df_tmp.duplicated()
n_dup = int(dup_mask.sum())
if n_dup > 0:
    print(f"Found {n_dup} duplicate samples in training set -> removing them.")
    train_df_tmp = train_df_tmp.loc[~dup_mask].reset_index(drop=True)
    y_train = train_df_tmp["y"].to_numpy(dtype=int)
    X_train = train_df_tmp.drop(columns=["y"]).to_numpy(dtype=float)
    print("New X_train:", X_train.shape, "New y_train:", y_train.shape)
else:
    print("No duplicate samples found in training set (based on selected features + label).")

# Standardize only continuous selected features
continuous_mask = np.array([
    len(np.unique(X_train[:, j])) > 2 for j in range(X_train.shape[1])
], dtype=bool)

X_train_std, X_test_std, means, stds = standardize_train_test(X_train, X_test, continuous_mask)

# Plot histograms of standardized continuous features
plot_histograms(X_train_std, selected_cols, continuous_mask, out_prefix="hist")
print("Saved histogram PNGs for standardized selected continuous features (files starting with hist_...).")


After encoding:
X shape: (45211, 45) y shape: (45211,)
Num features after encoding: 45
Top-7 features by |correlation| with target:
  duration                       corr=+0.3945
  poutcome_success               corr=+0.3068
  poutcome_unknown               corr=-0.1671
  contact_unknown                corr=-0.1509
  housing                        corr=-0.1392
  contact_cellular               corr=+0.1359
  month_mar                      corr=+0.1295
Stratified split shapes:
X_train: (36169, 7)
X_test : (9042, 7)
y_train: (36169,)
y_test : (9042,)
Found 27197 duplicate samples in training set -> removing them.
New X_train: (8972, 7) New y_train: (8972,)
Saved histogram PNGs for standardized selected continuous features (files starting with hist_...).


## Step C - Training Experiments (HW 2.3 Steps 2 and 3)

**Why this cell exists:** HW asks for cost-per-iteration plotting and multiple hyperparameter settings.

This cell:
- trains all combinations of learning rate and epoch count,
- stores per-epoch loss for each run,
- compares convergence speed and final performance,
- selects the best model by test F1.


In [32]:
all_loss_curves = []
experiment_rows = []
best_model = None
best_test_f1 = -1.0
best_label = None

for epochs in EPOCHS_LIST:
    for lr in LEARNING_RATES:
        model = LogisticRegressionScratch(lr=lr, epochs=epochs)
        losses = model.fit(X_train_std, y_train)

        label = f"lr={lr}, epochs={epochs}"
        all_loss_curves.append((label, losses))

        # Evaluate train and test for comparison across hyperparameters
        y_pred_train_i = model.predict(X_train_std, threshold=0.5)
        y_pred_test_i = model.predict(X_test_std, threshold=0.5)

        acc_tr_i, prec_tr_i, rec_tr_i, f1_tr_i, _ = classification_metrics(y_train, y_pred_train_i)
        acc_te_i, prec_te_i, rec_te_i, f1_te_i, _ = classification_metrics(y_test, y_pred_test_i)
        conv_epoch_90pct = convergence_epoch_to_90pct(losses)

        experiment_rows.append({
            "lr": float(lr),
            "epochs": int(epochs),
            "conv_epoch_90pct": int(conv_epoch_90pct),
            "final_train_loss": float(losses[-1]),
            "train_accuracy": float(acc_tr_i),
            "train_precision": float(prec_tr_i),
            "train_recall": float(rec_tr_i),
            "train_f1": float(f1_tr_i),
            "test_accuracy": float(acc_te_i),
            "test_precision": float(prec_te_i),
            "test_recall": float(rec_te_i),
            "test_f1": float(f1_te_i),
        })

        if f1_te_i > best_test_f1:
            best_test_f1 = f1_te_i
            best_model = model
            best_label = label

        print(f"Trained {label}")
        print("  Final training loss:", float(losses[-1]))
        print("  Convergence speed (epoch reaching 90% of total loss drop):", int(conv_epoch_90pct))
        print(f"  Test metrics  -> Acc={acc_te_i:.4f}, Prec={prec_te_i:.4f}, Rec={rec_te_i:.4f}, F1={f1_te_i:.4f}")
        print("  Model params (bias first):")
        for name, val in zip(["bias"] + selected_cols, model.w.tolist()):
            print(f"    {name:30s} {val:+.6f}")

exp_df = pd.DataFrame(experiment_rows)
exp_df = exp_df.sort_values(
    by=["test_f1", "test_accuracy", "conv_epoch_90pct"],
    ascending=[False, False, True]
).reset_index(drop=True)

summary_cols = [
    "lr", "epochs", "conv_epoch_90pct", "final_train_loss",
    "train_f1", "test_accuracy", "test_precision", "test_recall", "test_f1"
]
summary_df = exp_df[summary_cols].copy()
for c in ["final_train_loss", "train_f1", "test_accuracy", "test_precision", "test_recall", "test_f1"]:
    summary_df[c] = summary_df[c].round(4)

print("=== Hyperparameter comparison: convergence speed + final performance ===")
print("(Lower conv_epoch_90pct means faster convergence.)")
print(summary_df.to_string(index=False))

plot_loss_curves(all_loss_curves, out_path="loss_curves.png")
print("Saved loss curves plot: loss_curves.png")


Trained lr=0.1, epochs=300
  Final training loss: 0.5775891022848088
  Convergence speed (epoch reaching 90% of total loss drop): 114
  Test metrics  -> Acc=0.8996, Prec=0.6494, Rec=0.3081, F1=0.4179
  Model params (bias first):
    bias                           -0.450150
    duration                       +0.593139
    poutcome_success               +0.488473
    poutcome_unknown               -0.325128
    contact_unknown                -0.455790
    housing                        -0.474412
    contact_cellular               +0.439323
    month_mar                      +0.187601
Trained lr=0.05, epochs=300
  Final training loss: 0.5848569520248595
  Convergence speed (epoch reaching 90% of total loss drop): 159
  Test metrics  -> Acc=0.8955, Prec=0.6548, Rec=0.2259, F1=0.3359
  Model params (bias first):
    bias                           -0.366465
    duration                       +0.532075
    poutcome_success               +0.287787
    poutcome_unknown               -0.326743
 

## Step D - Final Reporting (HW 2.3 Steps 4, 5, 6)

**Why this cell exists:** It produces the final explainable report required in HW 2.3.

This cell reports:
1. learned parameters (including bias) and their role,
2. training and test metrics with definitions,
3. test confusion matrix and why it matters.


In [33]:
print("Best model by test F1:", best_label, f"(F1={best_test_f1:.4f})")

print("=== Best model parameters (bias first) ===")
for name, val in zip(["bias"] + selected_cols, best_model.w.tolist()):
    print(f"  {name:30s} {val:+.6f}")

explain_parameter_roles(selected_cols, best_model.w, continuous_mask)

y_pred_train = best_model.predict(X_train_std, threshold=0.5)
y_pred_test = best_model.predict(X_test_std, threshold=0.5)

acc_tr, prec_tr, rec_tr, f1_tr, _ = classification_metrics(y_train, y_pred_train)
acc_te, prec_te, rec_te, f1_te, cm_te = classification_metrics(y_test, y_pred_test)

explain_metric_definitions()

print("=== Training metrics ===")
print(f"Accuracy : {acc_tr:.4f}")
print(f"Precision: {prec_tr:.4f}")
print(f"Recall   : {rec_tr:.4f}")
print(f"F1-score : {f1_tr:.4f}")

print("=== Test metrics ===")
print(f"Accuracy : {acc_te:.4f}")
print(f"Precision: {prec_te:.4f}")
print(f"Recall   : {rec_te:.4f}")
print(f"F1-score : {f1_te:.4f}")

print("Confusion Matrix (Test) [[TN, FP],[FN, TP]]:", cm_te)
explain_confusion_matrix_importance(cm_te)

plot_confusion_matrix(cm_te, out_path="confusion_matrix.png")
print("Saved confusion matrix plot: confusion_matrix.png")


Best model by test F1: lr=0.1, epochs=600 (F1=0.4363)
=== Best model parameters (bias first) ===
  bias                           -0.595911
  duration                       +0.616485
  poutcome_success               +0.742675
  poutcome_unknown               -0.230891
  contact_unknown                -0.491358
  housing                        -0.525095
  contact_cellular               +0.541155
  month_mar                      +0.332769
Parameter-role interpretation (logistic model):
  logit(p) = bias + sum(w_i * x_i)
  bias = -0.595911: baseline log-odds when all input features are 0.
  - duration: w=+0.616485 -> +1 standardized unit (= +1 std in the original feature) increases log-odds; odds are multiplied by 1.852.
  - poutcome_success: w=+0.742675 -> +1 encoded unit increases log-odds; odds are multiplied by 2.102.
  - poutcome_unknown: w=-0.230891 -> +1 encoded unit decreases log-odds; odds are multiplied by 0.794.
  - contact_unknown: w=-0.491358 -> +1 encoded unit decreases log-